# STAGE A2 V1.5 — GOOGLE COLAB CANONICAL EXECUTION ENVIRONMENT QUALIFICATION
**Protocol**: Stage A2 Protocol V1.5 (Amendment 12 Locked)  
**Execution Scope**: 35,000 Train Sessions (586,577 events) | 7,500 Val Sessions (119,531 events)  
**Durable Root**: `/content/drive/MyDrive/Chuyende-stage-a2`  
**Strict Invariants**:
- `REAL_HDFS_COLAB_RUNS = 0` (During Qualification)
- `REAL_HDFS_COLAB_OPTIMIZER_STEPS = 0`
- `TEST_OPENED = false`
- `NON_EMPIRICAL_TEST_FIXTURE = true`


In [ ]:
# CELL 1 — Hosted Runtime / GPU Discovery (Discovery Only)
import subprocess, sys

print('=================================================================')
print('   GOOGLE COLAB HOSTED RUNTIME DISCOVERY                        ')
print('=================================================================')

try:
    smi_out = subprocess.check_output(['nvidia-smi'], text=True)
    print('NVIDIA Driver & GPU Detected:')
    print(smi_out.strip())
except Exception as e:
    raise RuntimeError(
        'FATAL: nvidia-smi failed! No GPU detected. '
        'Please select a GPU runtime in Colab: Runtime -> Change runtime type -> T4 GPU / L4 GPU / A100 GPU.'
    ) from e

py_ver = sys.version
print()
print('Host Python Version:', py_ver)

# Query GPU device properties via dedicated subprocess without importing torch in kernel
gpu_query = subprocess.check_output([
    'nvidia-smi', '--query-gpu=name,compute_cap,memory.total', '--format=csv,noheader'
], text=True).strip()
print('Assigned GPU Hardware:', gpu_query)

probe_cmd = [
    sys.executable, '-c',
    'import torch; print(f"Default Torch: {torch.__version__}, CUDA: {torch.version.cuda}, Available: {torch.cuda.is_available()}")'
]
probe_res = subprocess.run(probe_cmd, capture_output=True, text=True)
if probe_res.returncode == 0:
    print(probe_res.stdout.strip())
else:
    print('Torch probe note:', probe_res.stderr.strip())

print('Discovery Complete. Ready for Drive Mount & Source Checkout.')


In [ ]:
# CELL 2 — Mount Google Drive & Verify Durable Storage Directory
import os, sys
from pathlib import Path
from google.colab import drive

drive_mount_point = Path('/content/drive')
drive.mount(str(drive_mount_point), force_remount=False)

durable_root = Path('/content/drive/MyDrive/Chuyende-stage-a2')
durable_root.mkdir(parents=True, exist_ok=True)
assert durable_root.exists(), f'FATAL: Durable Google Drive directory missing: {durable_root}'
print('Google Drive Mounted Successfully:', durable_root)


In [ ]:
# CELL 3 — Clean Fresh Clone & Detached Checkout of Approved Commit
import os, subprocess, sys, shutil, re
from pathlib import Path

# RUNTIME PLACEHOLDER: The user will replace this with the final verified 40-hex commit SHA
APPROVED_PREPARATION_COMMIT = "<supplied-after-independent-review>"

repo_dir = Path('/content/Research')
repo_url = 'https://github.com/Minhlike/Chuyende.git'

# Validate 40-hex commit SHA before performing clone/checkout
if APPROVED_PREPARATION_COMMIT == '<supplied-after-independent-review>' or not re.match(r'^[0-9a-fA-F]{40}$', APPROVED_PREPARATION_COMMIT.strip()):
    raise RuntimeError(
        f"FATAL: APPROVED_PREPARATION_COMMIT is not set to a valid 40-character hex commit SHA! Current value: '{APPROVED_PREPARATION_COMMIT}'"
    )

# Always perform a clean fresh clone to avoid dirty residue
if repo_dir.exists():
    print(f'Removing existing {repo_dir} for fresh clean clone...')
    shutil.rmtree(repo_dir)

print(f'Cloning clean repository from {repo_url}...')
subprocess.run(['git', 'clone', repo_url, str(repo_dir)], check=True)

print(f'Detached checkout of approved commit: {APPROVED_PREPARATION_COMMIT}')
subprocess.run(['git', 'checkout', APPROVED_PREPARATION_COMMIT.strip()], cwd=str(repo_dir), check=True)

head_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=str(repo_dir), text=True).strip()
assert head_commit == APPROVED_PREPARATION_COMMIT.strip(), f'Commit mismatch! {head_commit} != {APPROVED_PREPARATION_COMMIT}'

# Verify clean tree
status_out = subprocess.check_output(['git', 'status', '--porcelain', 'src', 'scripts', 'experiments', 'tests'], cwd=str(repo_dir), text=True).strip()
assert len(status_out) == 0, f'Source tree is not clean! Diff:\n{status_out}'
print('Frozen Clean Source Verified at HEAD:', head_commit)


In [ ]:
# CELL 4 — Install Repository Dependencies & Exact PyTorch CUDA Runtime Check
import os, subprocess, sys
from pathlib import Path

repo_dir = Path('/content/Research')
print('Installing editable repository dependencies via pyproject.toml...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], cwd=str(repo_dir), check=True)

def verify_torch_runtime():
    test_cmd = [
        sys.executable, '-c',
        "import torch; "
        "assert torch.__version__ == '2.6.0+cu124', f'PyTorch version mismatch: {torch.__version__} != 2.6.0+cu124'; "
        "assert torch.version.cuda == '12.4', f'CUDA runtime mismatch: {torch.version.cuda} != 12.4'; "
        "assert torch.cuda.is_available() is True, 'CUDA not available in PyTorch!'"
    ]
    return subprocess.run(test_cmd, capture_output=True, text=True)

res = verify_torch_runtime()
if res.returncode != 0:
    print('Runtime PyTorch version is not 2.6.0+cu124. Reinstalling official PyTorch 2.6.0+cu124 wheel...')
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--force-reinstall',
        'torch==2.6.0', '--index-url', 'https://download.pytorch.org/whl/cu124'
    ], check=True)
    res_re = verify_torch_runtime()
    if res_re.returncode != 0:
        raise RuntimeError(f'FATAL: PyTorch verification failed after reinstall:\n{res_re.stderr}')

print('Exact PyTorch Runtime Verified: torch 2.6.0+cu124 (CUDA 12.4, cuda.is_available=True)')


In [ ]:
# CELL 5 — Fail-Closed Dataset Streaming SHA-256 Copy & Verification
import os, shutil, hashlib
from pathlib import Path

def compute_sha256_streaming(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    hasher = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(chunk_size):
            hasher.update(chunk)
    return hasher.hexdigest()

EXPECTED_HDFS_SHA = '6ca6c5bc2671c66afecee9369a2fdac606bf33997a2494ac66aa411fe3e95169'

drive_canonical = Path('/content/drive/MyDrive/Chuyende-stage-a2/datasets/HDFS_1.tar.gz')
drive_fallback = Path('/content/drive/MyDrive/HDFS_1.tar.gz')

if drive_canonical.exists():
    drive_source = drive_canonical
elif drive_fallback.exists():
    drive_source = drive_fallback
else:
    raise FileNotFoundError(
        f'FATAL: HDFS raw tarball missing on Drive! Checked:\n  - {drive_canonical}\n  - {drive_fallback}'
    )

print(f'Drive Dataset Source Found: {drive_source}')
src_sha = compute_sha256_streaming(drive_source)
print(f'Drive Source SHA-256: {src_sha}')
assert src_sha == EXPECTED_HDFS_SHA, f'Drive source SHA mismatch: {src_sha} != {EXPECTED_HDFS_SHA}'

local_dest = Path('/content/stage-a2-data/HDFS_1.tar.gz')
local_dest.parent.mkdir(parents=True, exist_ok=True)
tmp_dest = Path('/content/stage-a2-data/HDFS_1.tar.gz.tmp')

print(f'Copying Drive source to temporary local file: {tmp_dest}...')
shutil.copy2(drive_source, tmp_dest)
tmp_sha = compute_sha256_streaming(tmp_dest)
print(f'Temporary Local SHA-256: {tmp_sha}')
assert tmp_sha == EXPECTED_HDFS_SHA, f'Temporary file SHA mismatch: {tmp_sha} != {EXPECTED_HDFS_SHA}'

# Atomic replace
os.replace(tmp_dest, local_dest)
assert local_dest.exists(), f'Local destination missing at {local_dest}'

final_local_sha = compute_sha256_streaming(local_dest)
print(f'Canonical Local SHA-256: {final_local_sha}')
assert final_local_sha == EXPECTED_HDFS_SHA, f'Final local SHA mismatch: {final_local_sha} != {EXPECTED_HDFS_SHA}'
assert local_dest.stat().st_size == drive_source.stat().st_size, 'File size mismatch between Drive and local copy!'
print('HDFS Raw Dataset Parity Verified: 100% MATCH (6ca6c5bc2671c66a...)')


In [ ]:
# CELL 6 — Run Colab Bootstrap & Dynamic Hardware Environment Lock Generation
import os, subprocess, sys
from pathlib import Path

repo_dir = Path('/content/Research')
env_lock_output = repo_dir / 'experiments' / 'evidence' / 'stage-a2' / 'preexecution' / 'STAGE-A2-COLAB-EXECUTION-ENVIRONMENT-V1.5.json'
local_data = Path('/content/stage-a2-data/HDFS_1.tar.gz')
durable_root = Path('/content/drive/MyDrive/Chuyende-stage-a2')

assert repo_dir.exists(), f'Repo missing: {repo_dir}'
assert local_data.exists(), f'Dataset missing: {local_data}'
assert durable_root.exists(), f'Durable root missing: {durable_root}'

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

bootstrap_cmd = [
    sys.executable, 'scripts/bootstrap_stage_a2_colab.py',
    '--repo-dir', str(repo_dir),
    '--local-data-dest', str(local_data),
    '--durable-root', str(durable_root),
    '--env-lock-output', str(env_lock_output)
]

print('Running Bootstrap Subprocess:', ' '.join(bootstrap_cmd))
subprocess.run(bootstrap_cmd, cwd=str(repo_dir), check=True)

assert env_lock_output.exists(), f'FATAL: Environment lock file was not generated at {env_lock_output}'
print('Environment Lock Candidate Generated Successfully:', env_lock_output)


In [ ]:
# CELL 7 — Run NON_EMPIRICAL CUDA Deterministic Qualification
import os, subprocess, sys, json, hashlib
from pathlib import Path

def compute_sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

repo_dir = Path('/content/Research')
env_lock_path = repo_dir / 'experiments' / 'evidence' / 'stage-a2' / 'preexecution' / 'STAGE-A2-COLAB-EXECUTION-ENVIRONMENT-V1.5.json'
output_dir = repo_dir / 'experiments' / 'evidence' / 'stage-a2' / 'implementation'

assert env_lock_path.exists(), f'FATAL: Environment lock missing: {env_lock_path}'

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

qual_cmd = [
    sys.executable, 'scripts/run_stage_a2_deterministic_qualification.py',
    '--device', 'cuda',
    '--base-dir', str(repo_dir),
    '--environment-lock', str(env_lock_path),
    '--output-dir', str(output_dir)
]

print('Running Deterministic Qualification Subprocess:', ' '.join(qual_cmd))
subprocess.run(qual_cmd, cwd=str(repo_dir), check=True)

# Required artifact presence verification
required_artifacts = [
    'DETERMINISTIC-RESUME-EVIDENCE.json',
    'IMPLEMENTATION-QUALIFICATION.json',
    'ENVIRONMENT.json',
    'EXPERIMENTAL-SOURCE.json',
    'EVIDENCE-MANIFEST.json',
    'deterministic_resume.log',
    'qualification_checkpoint.pt'
]

for fname in required_artifacts:
    f_p = output_dir / fname
    assert f_p.exists(), f'FATAL: Required qualification artifact missing: {f_p}'

# Detailed verification
resume_p = output_dir / 'DETERMINISTIC-RESUME-EVIDENCE.json'
resume_data = json.loads(resume_p.read_text(encoding='utf-8'))
head_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=str(repo_dir), text=True).strip()
expected_lock_sha = compute_sha256(env_lock_path)

assert resume_data['qualification_status'] == 'PASS', f"Qualification status is not PASS: {resume_data['qualification_status']}"
assert resume_data['evidence_class'] == 'NON_EMPIRICAL_TEST_FIXTURE', f"Evidence class mismatch: {resume_data['evidence_class']}"
assert resume_data['fresh_process_isolated'] is True, 'Resume was not executed in isolated fresh process!'
assert resume_data['execution_code_commit_sha'] == head_commit, f"Commit mismatch: {resume_data['execution_code_commit_sha']} != {head_commit}"
assert resume_data['environment_lock_sha256'] == expected_lock_sha, f"Lock SHA mismatch: {resume_data['environment_lock_sha256']} != {expected_lock_sha}"

print('STAGE A2 DETERMINISTIC QUALIFICATION: PASS (100% Exact Numerical & Structural Identity)')


In [ ]:
# CELL 8 — Run Seed-42 Dry-Run Verification (Zero Optimizer Steps)
import os, subprocess, sys
from pathlib import Path

repo_dir = Path('/content/Research')
dataset_path = Path('/content/stage-a2-data/HDFS_1.tar.gz')
durable_runs = Path('/content/drive/MyDrive/Chuyende-stage-a2/runs')
plan_path = repo_dir / 'experiments' / 'plans' / 'STAGE-A2-FIVE-SEED-EXECUTION-PLAN-V1.5.json'
env_lock_path = repo_dir / 'experiments' / 'evidence' / 'stage-a2' / 'preexecution' / 'STAGE-A2-COLAB-EXECUTION-ENVIRONMENT-V1.5.json'
log_dest = repo_dir / 'experiments' / 'evidence' / 'stage-a2' / 'implementation' / 'SEED42-COLAB-DRY-RUN.log'

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

# DIRECT INVOCATION: Dedicated direct committed runner invocation without external wrappers
dry_cmd = [
    sys.executable, 'scripts/run_stage_a2_five_seed_empirical.py',
    '--seed', '42',
    '--dry-run',
    '--base-dir', str(repo_dir),
    '--dataset-path', str(dataset_path),
    '--durable-root', str(durable_runs),
    '--plan', str(plan_path),
    '--environment-lock', str(env_lock_path)
]

print('Executing Seed 42 Dry-Run Subprocess:', ' '.join(dry_cmd))
proc = subprocess.run(dry_cmd, cwd=str(repo_dir), capture_output=True, text=True)

# Capture combined stdout/stderr into evidence log
log_dest.parent.mkdir(parents=True, exist_ok=True)
combined_log = proc.stdout + '\n' + proc.stderr
log_dest.write_text(combined_log, encoding='utf-8')
print(proc.stdout)

if proc.returncode != 0:
    print('Dry-run STDERR:\n', proc.stderr)
    raise RuntimeError(f'FATAL: Seed 42 dry-run failed with code {proc.returncode}')

# Verification of dry run properties
assert 'OptimizerStepsExecuted=0' in combined_log or 'OptimizerStepsExecuted = 0' in combined_log or 'Optimizer Steps Executed: 0' in combined_log, 'Optimizer steps executed is NOT zero!'
assert 'TEST_OPENED=false' in combined_log or 'TEST_OPENED: false' in combined_log or 'Connected Test Firewall: LOCKED' in combined_log, 'Test firewall breach detected!'
assert 'Seed 42' in combined_log, 'Seed 42 not processed in dry-run!'
print('Seed 42 Dry-Run Evidence Verified: ZERO Optimizer Steps Executed (Log: ' + str(log_dest) + ')')


In [ ]:
# CELL 9 — Durably Mirror Qualification Artifacts to Google Drive
import os, subprocess, sys, json, hashlib, shutil
from datetime import datetime, timezone
from pathlib import Path

def compute_sha256_streaming(path: Path) -> str:
    hasher = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(8 * 1024 * 1024):
            hasher.update(chunk)
    return hasher.hexdigest()

repo_dir = Path('/content/Research')
durable_root = Path('/content/drive/MyDrive/Chuyende-stage-a2')

# Set canonical qualification identity variables in runtime globals
QUALIFICATION_RUN_ID = f"QUAL-COLAB-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
QUALIFICATION_DIR = durable_root / 'qualification' / QUALIFICATION_RUN_ID
FINAL_MANIFEST_PATH = QUALIFICATION_DIR / 'FINAL-QUALIFICATION-MANIFEST.json'

QUALIFICATION_DIR.mkdir(parents=True, exist_ok=True)

# Fail-closed host nvidia-smi capture (NO try/except error swallowing)
nvidiasmi_dest = repo_dir / 'experiments' / 'evidence' / 'stage-a2' / 'implementation' / 'NVIDIA-SMI.txt'
smi_out = subprocess.check_output(['nvidia-smi'], text=True)
nvidiasmi_dest.write_text(smi_out, encoding='utf-8')

files_to_mirror = [
    ('experiments/evidence/stage-a2/preexecution/STAGE-A2-COLAB-EXECUTION-ENVIRONMENT-V1.5.json', 'STAGE-A2-COLAB-EXECUTION-ENVIRONMENT-V1.5.json'),
    ('experiments/evidence/stage-a2/implementation/IMPLEMENTATION-QUALIFICATION.json', 'IMPLEMENTATION-QUALIFICATION.json'),
    ('experiments/evidence/stage-a2/implementation/DETERMINISTIC-RESUME-EVIDENCE.json', 'DETERMINISTIC-RESUME-EVIDENCE.json'),
    ('experiments/evidence/stage-a2/implementation/ENVIRONMENT.json', 'ENVIRONMENT.json'),
    ('experiments/evidence/stage-a2/implementation/EXPERIMENTAL-SOURCE.json', 'EXPERIMENTAL-SOURCE.json'),
    ('experiments/evidence/stage-a2/implementation/EVIDENCE-MANIFEST.json', 'EVIDENCE-MANIFEST.json'),
    ('experiments/evidence/stage-a2/implementation/deterministic_resume.log', 'deterministic_resume.log'),
    ('experiments/evidence/stage-a2/implementation/qualification_checkpoint.pt', 'qualification_checkpoint.pt'),
    ('experiments/evidence/stage-a2/implementation/SEED42-COLAB-DRY-RUN.log', 'SEED42-COLAB-DRY-RUN.log'),
    ('experiments/evidence/stage-a2/implementation/NVIDIA-SMI.txt', 'NVIDIA-SMI.txt')
]

manifest_entries = []
print(f'Mirroring qualification package to: {QUALIFICATION_DIR}')

for rel_src, rel_dst in files_to_mirror:
    src_p = repo_dir / rel_src
    assert src_p.exists(), f'FATAL: Required qualification artifact missing at source: {src_p}'
    dst_p = QUALIFICATION_DIR / rel_dst
    shutil.copy2(src_p, dst_p)
    
    src_sha = compute_sha256_streaming(src_p)
    dst_sha = compute_sha256_streaming(dst_p)
    assert src_sha == dst_sha, f'Mirror SHA mismatch for {rel_dst}: {dst_sha} != {src_sha}'
    
    manifest_entries.append({
        'name': rel_dst,
        'sha256': src_sha,
        'size_bytes': src_p.stat().st_size
    })
    print(f'  [MIRROR] {rel_dst} -> MATCH ({src_sha[:16]}...)')

final_manifest = {
    'qualification_run_id': QUALIFICATION_RUN_ID,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'storage': 'GOOGLE_DRIVE_DURABLE',
    'qualification_directory': str(QUALIFICATION_DIR),
    'artifact_count': len(manifest_entries),
    'artifacts': manifest_entries
}

FINAL_MANIFEST_PATH.write_text(json.dumps(final_manifest, indent=2) + '\n', encoding='utf-8')
print(f'Final Qualification Manifest Written: {FINAL_MANIFEST_PATH} (Artifacts: {len(manifest_entries)})')


In [ ]:
# CELL 10 — Final Qualification Hard Stop Gate & Verification Banner
import os, sys, json, hashlib
from pathlib import Path

def compute_sha256_streaming(path: Path) -> str:
    hasher = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(8 * 1024 * 1024):
            hasher.update(chunk)
    return hasher.hexdigest()

# 1. Verify exact runtime binding to Cell 9 execution
if 'QUALIFICATION_DIR' not in globals() or 'QUALIFICATION_RUN_ID' not in globals() or 'FINAL_MANIFEST_PATH' not in globals():
    raise RuntimeError('FATAL: Cell 9 did not complete in this runtime session! (QUALIFICATION_DIR missing from globals)')

current_qual_dir = Path(QUALIFICATION_DIR)
current_manifest_path = Path(FINAL_MANIFEST_PATH)
repo_dir = Path('/content/Research')

assert current_qual_dir.exists(), f'FATAL: Current qualification directory missing on Drive: {current_qual_dir}'
assert current_manifest_path.exists(), f'FATAL: FINAL-QUALIFICATION-MANIFEST.json missing at: {current_manifest_path}'
assert current_manifest_path == current_qual_dir / 'FINAL-QUALIFICATION-MANIFEST.json', 'Manifest path mismatch!'

# 2. Replay and verify FINAL-QUALIFICATION-MANIFEST.json
manifest_data = json.loads(current_manifest_path.read_text(encoding='utf-8'))
assert manifest_data['qualification_run_id'] == QUALIFICATION_RUN_ID, f"Run ID mismatch: {manifest_data['qualification_run_id']} != {QUALIFICATION_RUN_ID}"
assert manifest_data['storage'] == 'GOOGLE_DRIVE_DURABLE', f"Storage is not GOOGLE_DRIVE_DURABLE: {manifest_data['storage']}"
assert manifest_data['qualification_directory'] == str(current_qual_dir), f"Directory mismatch: {manifest_data['qualification_directory']} != {current_qual_dir}"

REQUIRED_ARTIFACT_NAMES = {
    'STAGE-A2-COLAB-EXECUTION-ENVIRONMENT-V1.5.json',
    'IMPLEMENTATION-QUALIFICATION.json',
    'DETERMINISTIC-RESUME-EVIDENCE.json',
    'ENVIRONMENT.json',
    'EXPERIMENTAL-SOURCE.json',
    'EVIDENCE-MANIFEST.json',
    'deterministic_resume.log',
    'qualification_checkpoint.pt',
    'SEED42-COLAB-DRY-RUN.log',
    'NVIDIA-SMI.txt'
}

manifest_names = {art['name'] for art in manifest_data['artifacts']}
assert manifest_names == REQUIRED_ARTIFACT_NAMES, f'Manifest artifact set mismatch! Difference: {manifest_names ^ REQUIRED_ARTIFACT_NAMES}'

# 3. Replay exact byte sizes and streaming SHA-256 for all 10 Drive artifacts
print('Replaying SHA-256 hashes for all durable Drive qualification artifacts...')
for art in manifest_data['artifacts']:
    durable_file = current_qual_dir / art['name']
    assert durable_file.exists(), f"Durable artifact missing on Google Drive: {durable_file}"
    assert durable_file.stat().st_size == art['size_bytes'], f"Size mismatch for {art['name']}: {durable_file.stat().st_size} != {art['size_bytes']}"
    actual_sha = compute_sha256_streaming(durable_file)
    assert actual_sha == art['sha256'], f"SHA mismatch for {art['name']}: {actual_sha} != {art['sha256']}"
    print(f"  [REPLAY VERIFIED] {art['name']} -> {actual_sha[:16]}...")

# 4. Validate DETERMINISTIC-RESUME-EVIDENCE
resume_path = current_qual_dir / 'DETERMINISTIC-RESUME-EVIDENCE.json'
resume_data = json.loads(resume_path.read_text(encoding='utf-8'))
assert resume_data['qualification_status'] == 'PASS', f"Qualification status is not PASS: {resume_data['qualification_status']}"
assert resume_data['evidence_class'] == 'NON_EMPIRICAL_TEST_FIXTURE', f"Evidence class is not NON_EMPIRICAL_TEST_FIXTURE: {resume_data['evidence_class']}"
assert resume_data['fresh_process_isolated'] is True, 'Not fresh process isolated!'

# 5. Validate Seed 42 Dry-Run log
dry_run_log = current_qual_dir / 'SEED42-COLAB-DRY-RUN.log'
dry_text = dry_run_log.read_text(encoding='utf-8')
assert 'OptimizerStepsExecuted=0' in dry_text or 'OptimizerStepsExecuted = 0' in dry_text or 'Optimizer Steps Executed: 0' in dry_text, 'Dry run steps not zero!'

# 6. Validate canonical authorization file DOES NOT exist before independent review
canonical_auth_file = repo_dir / 'experiments' / 'evidence' / 'stage-a2' / 'preexecution' / 'SEED42-COLAB-LAUNCH-AUTHORIZATION-V1.5.json'
if canonical_auth_file.exists():
    raise RuntimeError(f'FATAL SECURITY VIOLATION: Unauthorized launch authorization file exists before independent review: {canonical_auth_file}')

# 7. Display Final Stop Gate Summary
print('\n' + '=' * 65)
print('   STAGE A2 COLAB QUALIFICATION COMPLETE                        ')
print('=' * 65)
print('DETERMINISTIC QUALIFICATION:     PASS')
print('EVIDENCE CLASS:                  NON_EMPIRICAL_TEST_FIXTURE')
print('REAL HDFS OPTIMIZER STEPS:       0')
print('REAL SEED-42 AUTHORIZATION:      NO')
print(f'DURABLE QUALIFICATION DIRECTORY: {current_qual_dir}')
print('STATUS:                          STOP — PENDING INDEPENDENT REVIEW')
print('=' * 65 + '\n')
